In [1]:
from pyspark.sql.session import SparkSession
from pyspark.sql.types import IntegerType


def get_spark_session(appName="default"):
    spark = (
        SparkSession
        .builder
        .appName(appName)
        .master("local[*]")
        .config("spark.ui.port","4141")
        .config("spark.port.maxRetries", "0")
        .getOrCreate()
    )
    spark.sparkContext.setLogLevel("ERROR")
    return spark

def stop_start(app_name):

    global spark

    if 'spark' in globals():
        try:
            spark.stop()
            print("Old Spark session successfully stopped.")
        except Exception as e:
             print(f"No active session to stop. Error: {e}")
    else:
        print("No existing 'spark' variable found.")

    # 3. Return the fresh session
    print("Starting new Spark session...")
    return get_spark_session(app_name)


spark = stop_start("working_with_different_data_type")

No existing 'spark' variable found.
Starting new Spark session...


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/suanki/Desktop/pyspark-env/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/11/09 11:20:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### User-Defined Functions

- . Spark will serialize the function on the driver and transfer it over the network to all executor processes.

#### When you use the function, there are essentially two different things that occur.
  - If the function is written in Scala or Java, you can use it within the Java Virtual Machine (JVM).
  - If the function is written in Python, something quite different happens. Spark starts a Python
process on the worker, serializes all of the data to a format that Python can understand
(remember, it was in the JVM earlier), executes the function row by row on that data in the
Python process, and then finally returns the results of the row operations to the JVM and Spark.

<img src="../data/pictures/worker_python_process_udf.png" width="500">
separate python worker process

In [2]:

from pyspark.sql.functions import split

data = [
    {"emp_id": 101, "name": "Alice", "department": "HR", "salary": 50000, "city":"Pune, Banglore, Gorakhpur"},
    {"emp_id": 102, "name": "Bob", "department": "Engineering", "salary": 85000, "city":"PrayagRaj, Kolkata, Tuscon"},
    {"emp_id": 103, "name": "Charlie", "department": "Sales", "salary": 65000, "city":"Texax, LA, New Yark"},
    {"emp_id": 104, "name": "David", "department": "Engineering", "salary": 90000, "city":"Pune, New Yark, Gorakhpur"},
    {"emp_id": 105, "name": "Charls", "department": "Cleaner", "salary": 54545, "city":"Pune, Aurangabad, None"},
    {"emp_id": 106, "name": "DeadPool", "department": "Saver", "salary": 438438748, "city": None},
]

df = spark.createDataFrame(data)
df.printSchema()
df.show(10, False)



root
 |-- city: string (nullable = true)
 |-- department: string (nullable = true)
 |-- emp_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- salary: long (nullable = true)



+--------------------------+-----------+------+--------+---------+
|city                      |department |emp_id|name    |salary   |
+--------------------------+-----------+------+--------+---------+
|Pune, Banglore, Gorakhpur |HR         |101   |Alice   |50000    |
|PrayagRaj, Kolkata, Tuscon|Engineering|102   |Bob     |85000    |
|Texax, LA, New Yark       |Sales      |103   |Charlie |65000    |
|Pune, New Yark, Gorakhpur |Engineering|104   |David   |90000    |
|Pune, Aurangabad, None    |Cleaner    |105   |Charls  |54545    |
|NULL                      |Saver      |106   |DeadPool|438438748|
+--------------------------+-----------+------+--------+---------+



In [8]:
from pyspark.sql.functions import udf
from pyspark.sql.types import IntegerType

double_salary = udf( lambda x : x + x, IntegerType() )

double_salary_df = (df.select("salary", double_salary("salary").alias("double_salary")))

double_salary_df.show(10)


+---------+-------------+
|   salary|double_salary|
+---------+-------------+
|    50000|       100000|
|    85000|       170000|
|    65000|       130000|
|    90000|       180000|
|    54545|       109090|
|438438748|    876877496|
+---------+-------------+



pyspark.sql.classic.dataframe.DataFrame

In [11]:
def double_salary(num:IntegerType)->IntegerType :
    return num + num

double_salary_udf = udf(double_salary)

df.select("salary", double_salary_udf("salary")).show(10)

+---------+---------------------+
|   salary|double_salary(salary)|
+---------+---------------------+
|    50000|               100000|
|    85000|               170000|
|    65000|               130000|
|    90000|               180000|
|    54545|               109090|
|438438748|            876877496|
+---------+---------------------+



In [17]:
# work with sql , we need to register this udf
spark.udf.register("double_the_salary", double_salary, IntegerType())
df.createOrReplaceTempView("df_table")

spark.sql("""select salary, double_the_salary(salary) as new_salary from df_table """).show()

+---------+----------+
|   salary|new_salary|
+---------+----------+
|    50000|    100000|
|    85000|    170000|
|    65000|    130000|
|    90000|    180000|
|    54545|    109090|
|438438748| 876877496|
+---------+----------+



In [18]:
spark.stop()